In [ ]:
# ============================================================
# MEDIVOICE — Gemma 4 Good Hackathon | github.com/hamnamgl/Medivoice
# Offline AI Health Copilot for Frontline Community Health Workers
# ============================================================
import os, sys, threading, time, requests
from IPython.display import display, HTML

print("NOTE: This Kaggle notebook is a reproducible demo environment.")
print("NOTE: True offline deployment runs the same MediVoice stack locally with Ollama on-device.")

# ── 1. Clone / update repo ──────────────────────────────────
REPO = "/kaggle/working/Medivoice"
if not os.path.exists(REPO):
    os.system(f"git clone https://github.com/hamnamgl/Medivoice.git {REPO}")
    print("✅ Repo cloned")
else:
    os.system(f"git -C {REPO} pull")
    print("✅ Repo updated")
sys.path.insert(0, REPO)
os.chdir(REPO)

# ── 2. Install Python dependencies ──────────────────────────
os.system("pip install -q ollama openai-whisper edge-tts requests pyngrok")
print("✅ Dependencies installed")

# ── 3. Install zstd (required by Ollama installer) ──────────
os.system("apt-get install -y -q zstd")
print("✅ zstd installed")

# ── 4. Install Ollama (only if binary missing) ───────────────
OLLAMA_BIN = "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA_BIN):
    print("⬇️  Installing Ollama...")
    ret = os.system("curl -fsSL https://ollama.com/install.sh | sh")
    if ret == 0 and os.path.exists(OLLAMA_BIN):
        print("✅ Ollama installed")
    else:
        print("❌ Ollama install failed — check logs")
else:
    print("✅ Ollama already installed — skipping")

# ── 5. Start Ollama server ──────────────────────────────────
def start_ollama():
    os.system("ollama serve > /tmp/ollama.log 2>&1")

threading.Thread(target=start_ollama, daemon=True).start()

print("⏳ Waiting for Ollama server to start...")
for i in range(30):
    try:
        r = requests.get("http://localhost:11434/", timeout=2)
        if r.status_code == 200:
            print("✅ Ollama server is running")
            break
    except:
        pass
    time.sleep(1)
else:
    print("❌ Ollama server did not start in time — check /tmp/ollama.log")

# ── 6. Pull model only if not already cached ─────────────────
MODEL = "gemma3:4b"

def model_is_cached(model_name):
    """Check via Ollama API if model is already pulled."""
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = r.json().get("models", [])
        return any(m.get("name", "").startswith(model_name.split(":")[0]) for m in models)
    except:
        return False

if model_is_cached(MODEL):
    print(f"✅ {MODEL} already cached — skipping download")
else:
    print(f"⬇️  Downloading {MODEL} — only once, cached after this...")
    os.system(f"ollama pull {MODEL}")
    print("✅ Model downloaded and cached")

# ── 7. Smoke test ────────────────────────────────────────────
print("🔍 Running smoke test...")
try:
    r = requests.post("http://localhost:11434/api/chat", json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Say exactly: Medi is ready"}],
        "stream": False,
        "options": {"num_predict": 20}
    }, timeout=60)
    reply = r.json().get("message", {}).get("content", "")
    print(f"✅ Model working — Medi: {reply}")
except Exception as e:
    print(f"❌ Ollama error: {e}")
    print("📄 Ollama log tail:")
    os.system("tail -20 /tmp/ollama.log")

# ── 8. Public URL for PWA (Option 1: ngrok) ─────────────────
from pyngrok import ngrok

public_url = ngrok.connect(11434, bind_tls=True)
OLLAMA_URL = public_url.public_url.rstrip("/") + "/api/chat"

print(f"🌐 Ollama Public URL: {OLLAMA_URL}")
print("👉 PWA mein yeh URL 'Custom Ollama URL' field mein paste karo")
print("⚠️ Demo mode note: Phone/browser still needs internet to reach this tunnel.")

# ── 9. Live consultation demo ────────────────────────────────
from app.core.function_caller import run_agent
from app.utils.local_db import init_db, get_stats
init_db()

tests = [
    ("English",    "Child has high fever for 3 days and not eating"),
    ("Roman Urdu", "Bachche ko 3 din se tez bukhar hai"),
    ("Hausa",      "Yaro yana da zazzabi tsawon kwanaki 3"),
    ("Tool Test",  "15kg child — paracetamol dosage?"),
    ("Tool Test",  "Nearest hospital in Punjab?"),
    ("Emergency",  "Patient is unconscious and not breathing"),
]

print("\n" + "=" * 58)
print("🏥  MEDIVOICE — LIVE CONSULTATION DEMO")
print("     Offline AI · Gemma 4 · Ollama · 22 Languages")
print("=" * 58)

history = []
for label, msg in tests:
    print(f"\n[{label}]")
    print(f"👤 CHW   : {msg}")
    result = run_agent(msg, history)
    history = result["history"]
    print(f"🤖 Medi  : {result['response']}")
    if result["tool_used"]:
        print(f"🔧 Tool  : {result['tool_used']}")
        print(f"📋 Result: {result['tool_result']}")
    print("-" * 40)

# ── 10. Stats ────────────────────────────────────────────────
s = get_stats()
print(f"\n📊 Session Stats — Total:{s['total_visits']} | Emergency:{s['emergencies']} | Refer:{s['referrals']} | Home:{s['home_care']}")

# ── 11. PWA link ─────────────────────────────────────────────
display(HTML(f"""
<div style="font-family:sans-serif;background:#1a1a2e;color:#e0e0e0;
     padding:24px;border-radius:12px;border:1px solid #2a4a6a;margin:16px 0;">
  <h2 style="color:#7eb8f7;margin-bottom:8px;">📱 MediVoice PWA — Install on Android</h2>
  <p style="color:#a0c0a0;font-size:0.85rem;margin-bottom:16px;">
    Kaggle notebook = reproducible demo. True offline mode runs locally with Ollama on-device.
  </p>
  <a href="https://hamnamgl.github.io/Medivoice" target="_blank"
     style="display:inline-block;background:#4ade80;color:#0f0f1a;
            padding:12px 32px;border-radius:24px;font-weight:bold;
            text-decoration:none;font-size:1rem;margin-bottom:20px;">
    🌐 Open MediVoice PWA
  </a>
  <div style="background:#0a1a0a;border-radius:8px;padding:14px;
              font-size:0.82rem;color:#86efac;border-left:3px solid #4ade80;">
    <b>Use with Kaggle + ngrok:</b><br><br>
    1. Open <b>https://hamnamgl.github.io/Medivoice</b><br>
    2. Paste this into <b>Custom Ollama URL</b><br>
    3. <code>{OLLAMA_URL}</code><br>
    4. Tap <b>Save URL</b> and start chatting ✅<br><br>
    <b>Important:</b> This mode still needs internet between your phone and Kaggle.
  </div>
  <div style="margin-top:14px;font-size:0.78rem;color:#7eb8f7;">
    🏆 Targeting: Ollama Prize · Health &amp; Sciences · Digital Equity tracks<br>
    📂 Repo: <a href="https://github.com/hamnamgl/Medivoice"
                style="color:#4ade80;">github.com/hamnamgl/Medivoice</a>
  </div>
</div>
"""))

print("\n🏁 MediVoice demo complete.")
print("📂 Repo : https://github.com/hamnamgl/Medivoice")
print("🌐 PWA  : https://hamnamgl.github.io/Medivoice")
print(f"🔗 Tunnel: {OLLAMA_URL}")
